# Summary Table of Experimental Models and Findings from Four ALS Studies Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL for full reproducibility and interface with the standard.

**Dataset Croissant schema URL:**

`https://sen.science/doi/10.71728/senscience.f9eg-2wzn/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.f9eg-2wzn/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` fields for exploration. This helps to identify how data is organized and which fields/columns are present for further inspection.

In [ ]:
# List all record sets with their @id and names
print("Available Record Sets:")
record_sets = [rs for rs in dataset.record_sets()]
for rs in record_sets:
    print(f"  @id: {rs['@id']}, name: {rs.get('name', 'N/A')}, description: {rs.get('description', 'N/A')}")

# For each record set, print its fields and columns
for rs in record_sets:
    print(f"\nFields for RecordSet @id {rs['@id']}:")
    for field in rs['field']:
        print(f"  - field @id: {field['@id']} | name: {field.get('name', 'N/A')} | dataType: {field.get('dataType', 'N/A')}")
    if 'column' in rs:
        print('Columns:')
        for col in rs['column']:
            print(f"    - column @id: {col['@id']} | name: {col.get('name', 'N/A')}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the `@id` fields from the previous cell to reference record sets and fields.

In [ ]:
# Prepare DataFrames from each record set
dataframes = {}

for rs in dataset.record_sets():
    rs_id = rs['@id']
    # Load all records for this record set
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"First 5 records of record set '{rs_id}':\n", df.head(), '\n')

print(f"Loaded DataFrame keys: {list(dataframes.keys())}")# For demo, pick the first record set id for further analysis below.if len(dataframes) > 0:    main_record_set_id = list(dataframes.keys())[0]    print(f"Using record set: {main_record_set_id}")
    print(f"Columns: {dataframes[main_record_set_id].columns.tolist()}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as checking text frequency, unique values for categorical fields, etc. Since this dataset is a summary table (primarily text/string columns per the schema), we demonstrate filtering, counting, and simple text operations by `@id`.

In [ ]:
# Example EDA: Analyze frequency of topics and models
df = dataframes[main_record_set_id]

# Find the ID of the 'Topic' and 'Model' fields as reported previously. Adjust these variables if needed.
# You can review the field and column @id values from the overview cell.# Let's try most likely field names (since this Croissant is summarizing ALS studies):topic_cols = [col for col in df.columns if 'topic' in col.lower()]model_cols = [col for col in df.columns if 'model' in col.lower()]
if topic_cols:
    topic_col = topic_cols[0]
    print(f"Topic column: {topic_col}")
    print(df[topic_col].value_counts())
else:
    print("No topic field detected.")

if model_cols:
    model_col = model_cols[0]
    print(f"Model column: {model_col}")
    print(df[model_col].value_counts())
else:
    print("No model field detected.")

# For filtering, select all studies related to 'Mouse' models, if present
if model_cols and model_col in df.columns:
    mouse_df = df[df[model_col].str.lower().str.contains('mouse')]
    print(f"Number of studies using Mouse model: {len(mouse_df)}")
    print(mouse_df.head())

## 5. Visualization
Visualize basic distributions for categorical fields (e.g., counts by Topic, Model, or Section Used).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the counts of entries by Topic
if topic_cols:
    plt.figure(figsize=(8,4))
    sns.countplot(y=df[topic_col], order=df[topic_col].value_counts().index)
    plt.title('Counts by Topic')
    plt.xlabel('Count')
    plt.ylabel('Topic')
    plt.tight_layout()
    plt.show()

# Plot counts by Model
if model_cols:
    plt.figure(figsize=(8,2))
    sns.countplot(y=df[model_col], order=df[model_col].value_counts().index)
    plt.title('Counts by Model')
    plt.xlabel('Count')
    plt.ylabel('Model')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrates:
- How to load tabular summary data on ALS experimental models and findings via the Croissant schema with `mlcroissant`.
- How to programmatically explore record sets using their `@id` values, and examine the available fields.
- Basic exploratory analysis and filtering on key fields (such as Topic and Model).
- Visualization of data distributions.

This dataset serves as a reference for literature-based ALS model and key outcome summaries. For more complex processing, you may map fields by @id to downstream processing scripts and use the flexible, standardized metadata for reproducible workflows.